In [6]:
import math
import numpy as np

In [3]:
raw_data = """
0 0
0 1
-1 2
2 0
3 0
4 -1
""".strip()
points = []
for line in raw_data.splitlines():
    x, y = map(int, line.split())
    points.append((x, y))
points

[(0, 0), (0, 1), (-1, 2), (2, 0), (3, 0), (4, -1)]

In [13]:
def normal(mean: np.ndarray, cov: np.ndarray, x: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    mean = np.asarray(mean, dtype=float)
    cov = np.asarray(cov, dtype=float)
    d = mean.shape[0]
    diff = x - mean
    inv = np.linalg.inv(cov)
    det = np.linalg.det(cov)
    norm_const = 1.0 / (np.power(2 * np.pi, d / 2) * np.sqrt(det))
    return float(norm_const * np.exp(-0.5 * diff.T @ inv @ diff))

def em(
    mean1: np.ndarray,
    mean2: np.ndarray,
    cov1: np.ndarray,
    cov2: np.ndarray,
    points: list[tuple[int, int]],
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    x = np.asarray(points, dtype=float)
    w1 = np.array([normal(mean1, cov1, xi) for xi in x])
    w2 = np.array([normal(mean2, cov2, xi) for xi in x])
    total = w1 + w2
    total = np.where(total == 0.0, 1e-12, total)
    w1 = w1 / total
    w2 = w2 / total

    sum_weight1 = w1.sum()
    sum_weight2 = w2.sum()
    sum_weight1 = sum_weight1 if sum_weight1 > 0 else 1e-12
    sum_weight2 = sum_weight2 if sum_weight2 > 0 else 1e-12

    new_mean1 = (w1[:, None] * x).sum(axis=0) / sum_weight1
    new_mean2 = (w2[:, None] * x).sum(axis=0) / sum_weight2
    centered1 = x - new_mean1
    centered2 = x - new_mean2
    new_cov1 = (centered1.T * w1) @ centered1 / sum_weight1
    new_cov2 = (centered2.T * w2) @ centered2 / sum_weight2


    return new_mean1, new_mean2, new_cov1, new_cov2

mean1 = np.array([0.0, 1.0])
mean2 = np.array([3.0, 0.0])
cov1 = np.eye(2)
cov2 = np.eye(2)
mean1, mean2, cov1, cov2 =em(mean1, mean2, cov1, cov2, points)

print(f"Mean 1: {mean1}")
print(f"Mean 2: {mean2}")
print(f"Cov 1: {cov1}")
print(f"Cov 2: {cov2}")

Mean 1: [-0.23890579  0.96508475]
Mean 2: [ 3.01536809 -0.34253577]
Cov 1: [[ 0.43914806 -0.41442341]
 [-0.41442341  0.67860434]]
Cov 2: [[ 0.72584761 -0.34693272]
 [-0.34693272  0.23007812]]
